<a href="https://colab.research.google.com/github/batrisyiasuh/batrisyia5842342-AnalyticsEngineering/blob/main/502_Practice%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How I'm tackling this practice

I'm setting a timer for **55 minutes** to work on this. I'll treat it like a real test, keeping track of my decisions and checking my data at every single step.

### My Strategy:
* Keep track of how I wrote my code (the prompts I used, or what I looked up).
* Prove my data is actually correct with real counts and comparisons.
* Explain my design choices in plain, simple English.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [23]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## My Playground: Quick ways I can query the data

I can use SQL or pandas here. I'll run these quick examples to make sure I understand how to pull rows and join tables before I start building the real pipeline.

In [24]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [25]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## The Goal

I need to build a weekly operations report. For each week, city, and cuisine type, I want to show:
1. Total successful orders
2. Net money earned (revenue)
3. How fast we delivered the food (average time in minutes)

I also need to highlight how trustworthy these numbers really are.

# 1. Exploring and Profiling the Raw Data

*Before writing any main code, I need to look closely at these CSVs. I'm going to check for duplicates, find missing values, and see if there are any weird records that could break my code later.*

### Step 1.1: Checking Unique Keys and Duplicate Rows

First, I'll check if the main ID columns (like `order_id`) are unique and don't have missing values. I also want to check if there are any exact duplicate rows copied in the tables.

In [26]:
# Expected Result: A report indicating whether primary keys are unique (no duplicates) and free of nulls, plus a count of fully duplicated rows in each table.
for name, df in tables.items():
    print(f"=== Table: {name} ===")
    print(f"Total Rows: {df.shape[0]:,}")
    print(f"Fully duplicated rows count: {df.duplicated().sum()}")

    # Determine PK column(s)
    if name == 'orders':
        pk = ['order_id']
    elif name == 'order_status_events':
        pk = ['event_id']
    elif name == 'order_lines':
        pk = ['order_id', 'line_no']
    elif name == 'payments':
        pk = ['payment_id']
    elif name == 'restaurants':
        pk = ['restaurant_id']
    elif name == 'customers':
        pk = ['customer_id']
    else:
        pk = []

    if pk:
        null_pks = df[pk].isnull().any(axis=1).sum()
        dup_pks = df.duplicated(subset=pk).sum()
        print(f"Primary Key column(s): {pk}")
        print(f"  - Nulls in PK: {null_pks}")
        print(f"  - Duplicate PKs: {dup_pks}")
    print()


=== Table: orders ===
Total Rows: 14,494
Fully duplicated rows count: 0
Primary Key column(s): ['order_id']
  - Nulls in PK: 0
  - Duplicate PKs: 0

=== Table: order_status_events ===
Total Rows: 56,812
Fully duplicated rows count: 0
Primary Key column(s): ['event_id']
  - Nulls in PK: 0
  - Duplicate PKs: 0

=== Table: order_lines ===
Total Rows: 36,120
Fully duplicated rows count: 0
Primary Key column(s): ['order_id', 'line_no']
  - Nulls in PK: 0
  - Duplicate PKs: 0

=== Table: payments ===
Total Rows: 16,423
Fully duplicated rows count: 30
Primary Key column(s): ['payment_id']
  - Nulls in PK: 0
  - Duplicate PKs: 30

=== Table: restaurants ===
Total Rows: 67
Fully duplicated rows count: 0
Primary Key column(s): ['restaurant_id']
  - Nulls in PK: 0
  - Duplicate PKs: 3

=== Table: customers ===
Total Rows: 4,200
Fully duplicated rows count: 0
Primary Key column(s): ['customer_id']
  - Nulls in PK: 0
  - Duplicate PKs: 0



### Step 1.2: Looking for Nulls and Outlier Numbers

Now I'll check which columns have missing values, and look at the spread of our numbers (fees, item prices, payment amounts) to make sure nothing looks impossible (like negative prices).

In [27]:
# Expected Result: Null count summary per table, and a detailed summary statistic description of all numeric columns in the datasets.
for name, df in tables.items():
    print(f"=== {name} Null Counts ===")
    display(df.isnull().sum().to_frame(name='null_count'))
    print()

# Numeric summaries
print("=== Numeric Columns Distribution ===")
numeric_cols_map = {
    'orders': ['delivery_fee'],
    'order_lines': ['quantity', 'unit_price'],
    'payments': ['amount']
}

for name, cols in numeric_cols_map.items():
    print(f"\nTable: {name}")
    display(tables[name][cols].describe())


=== orders Null Counts ===


,null_count
order_id,0
customer_id,0
restaurant_id,0
order_ts,0
delivery_fee,0



=== order_status_events Null Counts ===


,null_count
event_id,0
order_id,0
status,0
event_ts,0



=== order_lines Null Counts ===


...

,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000



Table: order_lines


,quantity,unit_price
count,36120.000000,36120.000000
mean,1.995903,7.211739
std,0.813985,3.131357
min,1.000000,2.000000
25%,1.000000,5.000000
50%,2.000000,8.000000
75%,3.000000,10.500000
max,3.000000,12.000000



Table: payments


,amount
count,16423.000000
mean,30.988618
std,29.080373
min,-127.490000
25%,14.490000
50%,31.490000
75%,49.990000
max,127.490000


### Step 1.3: Spotting Spelling and Casing Mistakes

Computers treat 'LEEDS' and 'Leeds' as totally different cities, which would mess up my groupings. I'll scan the text columns to find and fix any of these inconsistencies.

In [28]:
# Expected Result: Value counts for crucial categorical dimensions, highlighting potential casing/whitespace discrepancies (e.g. "LEEDS" vs "Leeds").
categorical_checks = [
    ('order_status_events', 'status'),
    ('payments', 'payment_type'),
    ('restaurants', 'cuisine'),
    ('restaurants', 'city'),
    ('customers', 'home_city')
]

for table_name, col in categorical_checks:
    print(f"=== Value Counts for {table_name}.{col} ===")
    val_counts = tables[table_name][col].value_counts(dropna=False)
    display(val_counts.to_frame())

    # Check for casing / spacing inconsistencies
    raw_unique = tables[table_name][col].dropna().unique()
    cleaned_unique = {s.strip().lower() for s in raw_unique if isinstance(s, str)}
    if len(raw_unique) != len(cleaned_unique):
        print(f"⚠️ Potential inconsistency found in {table_name}.{col}!")
        print(f"   Raw Unique Count: {len(raw_unique)}, Cleaned (lowercased & stripped) Unique Count: {len(cleaned_unique)}")
    print()


=== Value Counts for order_status_events.status ===


,count
status,
placed,14494
accepted,14154
picked_up,13745
delivered,13549
cancelled,870



=== Value Counts for payments.payment_type ===


,count
payment_type,
charge,14524
refund,1899



=== Value Counts for restaurants.cuisine ===


,count
cuisine,
Vegan,9
Thai,8
Chinese,8
Japanese,7
Indian,7
Italian,7
Burgers,7
Mexican,6
mexican,2


⚠️ Potential inconsistency found in restaurants.cuisine!
   Raw Unique Count: 15, Cleaned (lowercased & stripped) Unique Count: 8

=== Value Counts for restaurants.city ===


,count
city,
London,23
Manchester,17
Birmingham,12
Bristol,7
Leeds,5
LEEDS,1
MANCHESTER,1
LONDON,1


⚠️ Potential inconsistency found in restaurants.city!
   Raw Unique Count: 8, Cleaned (lowercased & stripped) Unique Count: 5

=== Value Counts for customers.home_city ===


,count
home_city,
London,1541
Manchester,857
Birmingham,697
Bristol,569
Leeds,536


### Step 1.4: Checking the Date Ranges

Our task is to report on orders from March 1st to June 30th, 2026. I'll double-check the first and last dates in our tables to see if any data spills outside this window.

In [29]:
# Expected Result: Range validation showing min and max timestamps, identifying any out-of-bounds orders relative to the 1 Mar to 30 Jun 2026 window.
timestamp_cols = {
    'orders': 'order_ts',
    'order_status_events': 'event_ts',
    'payments': 'paid_at',
    'customers': 'signup_date' # signup_date is a date, but let's check its range as well
}

start_bound = pd.to_datetime('2026-03-01', utc=True)
end_bound = pd.to_datetime('2026-06-30 23:59:59.999999', utc=True)

for table_name, col in timestamp_cols.items():
    print(f"=== Timestamp Check: {table_name}.{col} ===")
    # Parse to datetime
    series = pd.to_datetime(tables[table_name][col], utc=True, errors='coerce')
    print(f"Min: {series.min()}")
    print(f"Max: {series.max()}")

    if table_name != 'customers': # Focus out-of-bounds checks on the operational tables
        out_of_bounds = series[(series < start_bound) | (series > end_bound)]
        print(f"Out of bounds (Outside Mar-Jun 2026) count: {out_of_bounds.count()}")
    print()


=== Timestamp Check: orders.order_ts ===
Min: 2026-03-01 09:13:23+00:00
Max: 2026-06-30 22:58:46+00:00
Out of bounds (Outside Mar-Jun 2026) count: 0

=== Timestamp Check: order_status_events.event_ts ===
Min: 2026-03-01 09:13:23+00:00
Max: 2026-06-30 23:37:01+00:00
Out of bounds (Outside Mar-Jun 2026) count: 0

=== Timestamp Check: payments.paid_at ===
Min: 2026-03-01 09:14:35+00:00
Max: 2026-07-02 17:13:58+00:00
Out of bounds (Outside Mar-Jun 2026) count: 5

=== Timestamp Check: customers.signup_date ===
Min: 2024-01-02 00:00:00+00:00
Max: 2026-03-10 00:00:00+00:00



### Step 1.5: Verifying Table Relationships

I want to make sure the tables actually fit together perfectly. I'll check if we have any 'orphan' records (like a payment for an order that doesn't exist in our main orders table).

In [30]:
# Expected Result: Quantitative measures of how well child tables match parent records, spotting any orphan child entries or missing references in dimensions.
orders_df = tables['orders']

# Child counts per order_id
for child_name in ['order_status_events', 'order_lines', 'payments']:
    child_df = tables[child_name]
    grp = child_df.groupby('order_id').size()
    print(f"=== Distribution of {child_name} rows per order ===")
    print(f"  - Min: {grp.min()}")
    print(f"  - Avg: {grp.mean():.2f}")
    print(f"  - Max: {grp.max()}")

    # Orphan checks
    orphans = child_df[~child_df['order_id'].isin(orders_df['order_id'])]['order_id'].nunique()
    print(f"  - Distinct orphan order_ids (present in {child_name} but NOT in orders): {orphans}")

    # Orders missing this child
    missing_child = orders_df[~orders_df['order_id'].isin(child_df['order_id'])]['order_id'].nunique()
    print(f"  - Orders with NO records in {child_name}: {missing_child}")
    print()

# Dimension missingness checks
missing_cust = orders_df[~orders_df['customer_id'].isin(tables['customers']['customer_id'])]['customer_id'].nunique()
missing_rest = orders_df[~orders_df['restaurant_id'].isin(tables['restaurants']['restaurant_id'])]['restaurant_id'].nunique()
print(f"Distinct customer_ids in orders missing from customers table: {missing_cust}")
print(f"Distinct restaurant_ids in orders missing from restaurants table: {missing_rest}")

# Restaurant uniqueness / effective_from check
rest_df = tables['restaurants']
print(f"\nTotal rows in restaurants: {len(rest_df)}")
print(f"Distinct restaurant_ids: {rest_df['restaurant_id'].nunique()}")
multiple_versions = rest_df.groupby('restaurant_id').filter(lambda x: len(x) > 1)
if not multiple_versions.empty:
    print("\nExample of multiple restaurant records per ID (SCD check):")
    display(multiple_versions.sort_values('restaurant_id').head(10))


=== Distribution of order_status_events rows per order ===
  - Min: 2
  - Avg: 3.92
  - Max: 6
  - Distinct orphan order_ids (present in order_status_events but NOT in orders): 0
  - Orders with NO records in order_status_events: 0

=== Distribution of order_lines rows per order ===
  - Min: 1
  - Avg: 2.49
  - Max: 4
  - Distinct orphan order_ids (present in order_lines but NOT in orders): 0
  - Orders with NO records in order_lines: 0

=== Distribution of payments rows per order ===
  - Min: 1
  - Avg: 1.13
  - Max: 3
  - Distinct orphan order_ids (present in payments but NOT in orders): 0
  - Orders with NO records in payments: 0

Distinct customer_ids in orders missing from customers table: 0
Distinct restaurant_ids in orders missing from restaurants table: 0

Total rows in restaurants: 67
Distinct restaurant_ids: 64

Example of multiple restaurant records per ID (SCD check):


,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01


### Step 1.6: Deep-diving into Order Statuses

I need to understand what happens to an order. I'll look at how many orders got delivered vs. cancelled, and check if any orders have duplicate status updates (like being accepted twice).

In [31]:
# Expected Result: Count of orders with duplicate same-status events, unique status combinations for 5/6-event orders, and overlap between 'delivered' & 'cancelled' states.
status_events = tables['order_status_events']

# 1. Orders with more than one event of the same status
same_status_counts = status_events.groupby(['order_id', 'status']).size().reset_index(name='count')
dup_status_orders = same_status_counts[same_status_counts['count'] > 1]
print(f"Number of orders with more than one event of the same status: {dup_status_orders['order_id'].nunique()}")
if not dup_status_orders.empty:
    display(dup_status_orders.head())

# 2. Status combinations for orders with 5 or 6 events
event_counts_per_order = status_events.groupby('order_id').size()
complex_orders = event_counts_per_order[event_counts_per_order.isin([5, 6])].index

complex_paths = (
    status_events[status_events['order_id'].isin(complex_orders)]
    .sort_values(by=['order_id', 'event_ts'])
    .groupby('order_id')['status']
    .apply(lambda x: ' -> '.join(x))
    .reset_index(name='path')
)
print(f"\nUnique status paths for orders with 5 or 6 events:")
display(complex_paths['path'].value_counts().to_frame())

# 3. Delivered vs Cancelled overlaps
status_pivot = status_events.pivot_table(index='order_id', columns='status', aggfunc='size', fill_value=0)

both_status = status_pivot[(status_pivot['delivered'] > 0) & (status_pivot['cancelled'] > 0)]
neither_status = status_pivot[(status_pivot['delivered'] == 0) & (status_pivot['cancelled'] == 0)]

print(f"\nOrders with BOTH 'delivered' and 'cancelled': {len(both_status)}")
print(f"Orders with NEITHER 'delivered' nor 'cancelled': {len(neither_status)}")


Number of orders with more than one event of the same status: 357


,order_id,status,count
39,O000011,accepted,2
71,O000019,delivered,2
80,O000021,picked_up,2
701,O000181,accepted,2
924,O000238,accepted,2



Unique status paths for orders with 5 or 6 events:


,count
path,
placed -> accepted -> picked_up -> delivered -> delivered,138
placed -> accepted -> picked_up -> picked_up -> delivered,119
placed -> accepted -> accepted -> picked_up -> delivered,94
placed -> accepted -> accepted -> picked_up -> delivered -> delivered,2
placed -> accepted -> accepted -> picked_up -> picked_up -> delivered,1



Orders with BOTH 'delivered' and 'cancelled': 0
Orders with NEITHER 'delivered' nor 'cancelled': 215


### Step 1.7: Checking Time Sequences

I'll calculate the time between an order being placed and delivered. I need to make sure there aren't any time-traveling orders (where the delivery timestamp is earlier than the placement timestamp!).

In [32]:
# Expected Result: Count of out-of-sequence orders, descriptive statistics for fulfillment duration, and confirmation of alignment between 'placed' and 'order_ts'.
# 1. Pivot first timestamp per status
status_ts = status_events.copy()
status_ts['event_ts'] = pd.to_datetime(status_ts['event_ts'], utc=True)
earliest_status_ts = status_ts.sort_values('event_ts').groupby(['order_id', 'status'])['event_ts'].first().unstack()

# 2. Sequence checks
# Common sequence: placed -> accepted -> picked_up -> delivered
out_of_seq = earliest_status_ts[
    (earliest_status_ts['delivered'] < earliest_status_ts['placed']) |
    (earliest_status_ts['delivered'] < earliest_status_ts['accepted']) |
    (earliest_status_ts['picked_up'] < earliest_status_ts['accepted']) |
    (earliest_status_ts['accepted'] < earliest_status_ts['placed'])
]
print(f"Number of out-of-sequence orders: {len(out_of_seq)}")

# 3. Placed -> Delivered duration analysis
delivered_orders = earliest_status_ts[earliest_status_ts['delivered'].notnull() & earliest_status_ts['placed'].notnull()]
duration_min = (delivered_orders['delivered'] - delivered_orders['placed']).dt.total_seconds() / 60.0

print("\nPlaced to Delivered Duration in minutes:")
display(duration_min.describe())
print(f"Any negative durations?: {len(duration_min[duration_min < 0])}")

# 4. Check 'placed' event_ts matching orders.order_ts
orders_clean_ts = tables['orders'].copy()
orders_clean_ts['order_ts'] = pd.to_datetime(orders_clean_ts['order_ts'], utc=True)

comparison = earliest_status_ts[['placed']].merge(orders_clean_ts[['order_id', 'order_ts']], left_index=True, right_on='order_id')
mismatches = comparison[comparison['placed'] != comparison['order_ts']]
print(f"\nMismatches between 'placed' timestamp and orders.order_ts: {len(mismatches)}")


Number of out-of-sequence orders: 16

Placed to Delivered Duration in minutes:


,0
count,13409.000000
mean,36.402379
std,23.255764
min,-58.000000
25%,30.433333
50%,35.416667
75%,40.716667
max,1194.266667


Any negative durations?: 16

Mismatches between 'placed' timestamp and orders.order_ts: 0


### Step 1.8: Reconciling Cash and Order Prices

To make sure our payments data is correct, I'll calculate what each delivered order *should* have cost (item prices × quantity + delivery fee) and compare it directly to what the customer was actually charged.

In [33]:
# Expected Result: Breakdown of amounts/refunds, counts of orders with anomalies (e.g. refund exceeding charge), and discrepancy matches on calculated sums.
payments = tables['payments'].copy()

# 1. Confirm that 30 duplicate rows are all charges
dup_rows = payments[payments.duplicated(keep=False)]
print(f"Duplicate rows count: {len(dup_rows)}")
print("Duplicate rows payment types:")
print(dup_rows['payment_type'].value_counts())

# 2. Show sign of amount by payment_type
print("\nAmount range by payment type:")
display(payments.groupby('payment_type')['amount'].agg(['min', 'max']))

# 3. Multi-charge & Refund anomalies
pay_pivot = payments.pivot_table(index='order_id', columns='payment_type', values='amount', aggfunc='sum', fill_value=0.0)

multi_charge = payments[payments['payment_type'] == 'charge'].groupby('order_id').size()
orders_multi_charge = len(multi_charge[multi_charge > 1])
refund_exceed_charge = len(pay_pivot[pay_pivot['refund'] > pay_pivot['charge']])
orders_with_refund = pay_pivot[pay_pivot['refund'] > 0].index

# Link refunds to cancelled orders
cancelled_orders = status_pivot[status_pivot['cancelled'] > 0].index
refunded_and_cancelled = len(set(orders_with_refund).intersection(set(cancelled_orders)))

print(f"\nOrders with more than one distinct charge event: {orders_multi_charge}")
print(f"Orders with total refunds > total charges: {refund_exceed_charge}")
print(f"Orders with refunds: {len(orders_with_refund)}")
print(f"Of those, how many were cancelled: {refunded_and_cancelled}")

# 4. Check if charge amount matches expected sum on delivered orders
delivered_ids = status_pivot[status_pivot['delivered'] > 0].index

# Sum of order lines per order
order_lines = tables['order_lines'].copy()
order_lines['line_total'] = order_lines['quantity'] * order_lines['unit_price']
line_sums = order_lines.groupby('order_id')['line_total'].sum()

# Merge with order delivery fee
order_fees = tables['orders'].set_index('order_id')[['delivery_fee']]
expected_totals = line_sums.to_frame().join(order_fees)
expected_totals['expected_charge'] = expected_totals['line_total'] + expected_totals['delivery_fee']

# Compare with actual charge amount
actual_charges = payments[payments['payment_type'] == 'charge'].groupby('order_id')['amount'].sum()
reconciliation = expected_totals.join(actual_charges.to_frame(name='actual_charge'), how='inner')
reconciliation = reconciliation.loc[reconciliation.index.isin(delivered_ids)]

reconciliation['matches'] = (reconciliation['expected_charge'] - reconciliation['actual_charge']).abs() < 0.01
match_rate = reconciliation['matches'].mean() * 100
print(f"\nPricing match rate for delivered orders (charge == lines_sum + delivery_fee): {match_rate:.2f}%")


Duplicate rows count: 60
Duplicate rows payment types:
payment_type
charge    60
Name: count, dtype: int64

Amount range by payment type:


,min,max
payment_type,,
charge,2.00,127.49
refund,-127.49,-0.35



Orders with more than one distinct charge event: 30
Orders with total refunds > total charges: 0
Orders with refunds: 0
Of those, how many were cancelled: 0

Pricing match rate for delivered orders (charge == lines_sum + delivery_fee): 99.79%


### Step 1.9: Tracking Moving Restaurants (SCD Type 2)

Some restaurants actually moved cities mid-way through! I need to inspect these movers (`R007`, `R021`, and `R040`) and see exactly when they changed locations so I can assign their orders to the right city.

In [34]:
# Expected Result: Breakdown of orders before and after the change/effective date for the multi-version restaurants.
scd_restaurants = ['R007', 'R021', 'R040']
rest_scd = tables['restaurants'][tables['restaurants']['restaurant_id'].isin(scd_restaurants)].copy()
print("Slowly Changing Dimension table records:")
display(rest_scd.sort_values(by=['restaurant_id', 'effective_from']))

# Parse order timestamp to dates
orders_with_dates = tables['orders'][tables['orders']['restaurant_id'].isin(scd_restaurants)].copy()
orders_with_dates['order_date'] = pd.to_datetime(orders_with_dates['order_ts'], utc=True).dt.date

# Let's count orders relative to their relocation boundaries
# For example: R007 relocated on 2026-04-14, R021 on 2026-05-06, R040 on 2026-05-27
relocation_dates = {
    'R007': pd.to_datetime('2026-04-14').date(),
    'R021': pd.to_datetime('2026-05-06').date(),
    'R040': pd.to_datetime('2026-05-27').date(),
}

for r_id, cutoff in relocation_dates.items():
    subset = orders_with_dates[orders_with_dates['restaurant_id'] == r_id]
    before = subset[subset['order_date'] < cutoff]
    after = subset[subset['order_date'] >= cutoff]
    print(f"\nRestaurant {r_id} (Relocation boundary: {cutoff}):")
    print(f"  - Orders BEFORE relocation: {len(before)}")
    print(f"  - Orders AFTER relocation: {len(after)}")


Slowly Changing Dimension table records:


,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
48,R040,Corner Vegan 40,Vegan,London,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27



Restaurant R007 (Relocation boundary: 2026-04-14):
  - Orders BEFORE relocation: 23
  - Orders AFTER relocation: 31

Restaurant R021 (Relocation boundary: 2026-05-06):
  - Orders BEFORE relocation: 97
  - Orders AFTER relocation: 69

Restaurant R040 (Relocation boundary: 2026-05-27):
  - Orders BEFORE relocation: 97
  - Orders AFTER relocation: 41


Add text here

# 2. My Game Plan

*Before writing the main build code, here is how I'm going to structure the pipeline and handle the data issues I found.*

### **2.1 Key decisions**

1. **One row of the final table.** One week × cuisine × city, with order count, net revenue and average delivery minutes.
2. **Which week an order belongs to.** The week (starting Monday) in which the order was placed, using `orders.order_ts`. I will keep UTC for simplicity because the operational system and dataset are standardized in UTC, which avoids complex timezone conversions across standard and daylight savings boundaries. The first and last weeks are partial (1 and 2 days), so I will flag them in the trust notes.
3. **Which orders count.**
   - Delivered orders (those with a `delivered` event) count as orders in the main table.
   - Cancelled orders are excluded from the main table and added later as a separate column if time allows.
   - Orders with neither a delivered nor a cancelled event are treated as unfinished and excluded. I will report how many there are and when they were placed.
4. **Net revenue.** The sum of money received per order from payments, after removing the 30 exact duplicate rows. Since refund amounts in our data are negative, net revenue is the sum of all payment amounts for those orders. It includes the delivery fee, because charges reconcile with items plus fee in 99.79% of delivered orders (Step 1.8). Revenue is assigned to the order’s week, not the payment date, because 5 payments fall after 30 June.
5. **Delivery time.** Minutes from the placed event to the first delivered event. It can only be measured for delivered orders that have both events. The 16 orders where delivery comes before placement are excluded from the average and counted in the trust notes.
6. **Cleaning.**
   - Trim spaces and standardise casing in `cuisine` and `city` (e.g. `LEEDS` to `Leeds`). Afterwards I expect 8 cuisines and 5 cities.
   - Remove the 30 duplicate payments.
   - Restaurants `R007`, `R021` and `R040` each have two rows because they moved city. I will match each order to the version in effect when it was placed (the latest `effective_from` on or before `order_ts`), so each order gets exactly one restaurant row.

### **2.2 Join order and expected row counts**

| Step | Table | Expected rows |
| --- | --- | --- |
| 1 | Payments: 16,423 minus 30 duplicates | 16,393 |
| 2 | Payments grouped to one row per order | 14,494 |
| 3 | Events grouped to one row per order (placed, delivered, cancelled timestamps) | 14,494 |
| 4 | Restaurants cleaned, 64 distinct ids | 67 rows before matching |
| 5 | `silver_orders`: orders joined to steps 2, 3 and 4 | 14,494 after each join |
| 6 | Gold: delivered orders grouped by week, cuisine, city | at most about 760 rows |

*If any join in step 5 gives more than 14,494 rows, a fan-out has happened and I stop and investigate.*

### **2.3 How I will check the results**
- The gold order total equals the number of delivered orders in silver.
- Gold net revenue equals the cleaned payments total for the same orders.
- Report the share of delivered orders with a measurable delivery time.

# 3. Let's Build It!

*I'm going to build this in small, clean steps so I keep complete control over the data flow. If anything goes wrong, it will be super easy to spot.*

### **Step 3.1: Clean Payments**
We will remove the 30 exact duplicate rows, verify that we have exactly 16,393 rows, group payment amounts per order into `total_charged`, `total_refunded`, and `net_revenue`, and ensure we have exactly 14,494 unique orders.

In [35]:
# Expected Result:
# - Step 1 Payments deduplicated rows: 16,393
# - Step 2 Grouped payments row count: 14,494
import pandas as pd

payments_raw = tables['payments'].copy()

# 1. Remove 30 exact duplicates
payments_clean = payments_raw.drop_duplicates()
print(f"Cleaned payments row count: {len(payments_clean):,}")

# 2. Separate charges and refunds to pivot, summing up by order
# Since refunds are negative numbers, sum directly to calculate net_revenue
pay_grouped = payments_clean.groupby(['order_id', 'payment_type'])['amount'].sum().unstack(fill_value=0.0)
pay_grouped = pay_grouped.rename(columns={'charge': 'total_charged', 'refund': 'total_refunded'})

# Calculate net_revenue
pay_grouped['net_revenue'] = pay_grouped['total_charged'] + pay_grouped['total_refunded']
pay_grouped = pay_grouped.reset_index()

print(f"Grouped payments per order row count: {len(pay_grouped):,}")
display(pay_grouped.head(5))

Cleaned payments row count: 16,393
Grouped payments per order row count: 14,494


payment_type,order_id,total_charged,total_refunded,net_revenue
0,O000001,37.49,0.0,37.49
1,O000002,31.99,0.0,31.99
2,O000003,12.50,0.0,12.50
3,O000004,46.99,0.0,46.99
4,O000005,6.00,-6.0,0.00


### **Step 3.2: Aggregate Order Events**
Next, we will process the status event history to get the earliest timestamps for `placed`, `delivered`, and `cancelled` statuses per order.

In [36]:
# Expected Result: 14,494 rows, each order_id representing one row with parsed UTC timestamps for key stages.
events_raw = tables['order_status_events'].copy()
events_raw['event_ts'] = pd.to_datetime(events_raw['event_ts'], utc=True)

# Get earliest event timestamp per status per order
events_grouped = (
    events_raw.sort_values('event_ts')
    .groupby(['order_id', 'status'])['event_ts']
    .first()
    .unstack()
    .rename(columns={'placed': 'placed_ts', 'delivered': 'delivered_ts', 'cancelled': 'cancelled_ts'})
    .reset_index()
)

print(f"Grouped events per order row count: {len(events_grouped):,}")
display(events_grouped.head(5))

Grouped events per order row count: 14,494


status,order_id,accepted,cancelled_ts,delivered_ts,picked_up,placed_ts
0,O000001,2026-03-01 09:15:28+00:00,NaT,2026-03-01 10:13:39+00:00,2026-03-01 09:37:00+00:00,2026-03-01 09:13:23+00:00
1,O000002,2026-03-01 09:21:19+00:00,NaT,2026-03-01 10:01:53+00:00,2026-03-01 09:39:13+00:00,2026-03-01 09:15:33+00:00
2,O000003,2026-03-01 09:27:41+00:00,NaT,2026-03-01 10:02:45+00:00,2026-03-01 09:49:37+00:00,2026-03-01 09:23:42+00:00
3,O000004,2026-03-01 09:29:25+00:00,NaT,2026-03-01 10:10:41+00:00,2026-03-01 09:52:15+00:00,2026-03-01 09:28:14+00:00
4,O000005,2026-03-01 11:36:36+00:00,2026-03-01 11:41:33+00:00,NaT,NaT,2026-03-01 11:32:35+00:00


### **Step 3.3: Standardize Restaurants**
We will trim spaces and title-case the `cuisine` and `city` columns to eliminate spelling or casing issues (such as `LEEDS` vs `Leeds`), resulting in exactly 8 distinct cuisines and 5 distinct cities.

In [37]:
# Expected Result: Cleaned restaurants with 67 rows, having standardized cities (5 unique) and cuisines (8 unique).
restaurants_clean = tables['restaurants'].copy()

# Clean string values
restaurants_clean['cuisine'] = restaurants_clean['cuisine'].astype(str).str.strip().str.title()
restaurants_clean['city'] = restaurants_clean['city'].astype(str).str.strip().str.title()
restaurants_clean['effective_from'] = pd.to_datetime(restaurants_clean['effective_from'], utc=True)

print(f"Standardized cuisines: {restaurants_clean['cuisine'].nunique()}", list(restaurants_clean['cuisine'].unique()))
print(f"Standardized cities: {restaurants_clean['city'].nunique()}", list(restaurants_clean['city'].unique()))
print(f"Total cleaned restaurant rows: {len(restaurants_clean)}")

Standardized cuisines: 8 ['Vegan', 'Burgers', 'Japanese', 'Indian', 'Thai', 'Italian', 'Mexican', 'Chinese']
Standardized cities: 5 ['London', 'Bristol', 'Birmingham', 'Leeds', 'Manchester']
Total cleaned restaurant rows: 67


### **Step 3.4: Build Silver Layer (silver_orders)**
Now we will assemble our unified master `silver_orders` dataframe by performing chronological slowly changing dimension joins for the moving restaurants, merging clean payment summaries, and matching status transitions.

In [40]:
# Expected Result: Exactly 14,494 rows with no row fan-outs or duplicate expansion.
orders_base = tables['orders'].copy()
orders_base['order_ts'] = pd.to_datetime(orders_base['order_ts'], utc=True)

# --- 1. Map each order to the correct historical version of the restaurant (SCD Type 2) ---
# For each order, we find the restaurant row with the latest 'effective_from' that is on or before the 'order_ts'
merged_rest = pd.merge(orders_base, restaurants_clean, on='restaurant_id', how='left')
valid_rest_matches = merged_rest[merged_rest['order_ts'] >= merged_rest['effective_from']]

# Select the record with the maximum/latest effective_from date per order_id
rest_scd_mapped = (
    valid_rest_matches.sort_values('effective_from')
    .groupby('order_id')
    .last()
    .reset_index()
    .drop(columns=['effective_from'])
)

# Ensure all orders got matched (comma removed from 14494 to resolve the SyntaxError)
assert len(rest_scd_mapped) == 14494, f"SCD Mapping mismatch! Expected 14494, got {len(rest_scd_mapped)}"

# --- 2. Merge status events and aggregated payments ---
silver_orders = rest_scd_mapped.merge(events_grouped, on='order_id', how='left')
silver_orders = silver_orders.merge(pay_grouped, on='order_id', how='left')

# --- 3. Compute delivery time ---
# Calculate duration in minutes (earliest delivered_ts minus placed_ts)
silver_orders['delivery_time_mins'] = (silver_orders['delivered_ts'] - silver_orders['placed_ts']).dt.total_seconds() / 60.0

# Flag sequence anomalies where delivered timestamp preceded placement
silver_orders['is_delivery_invalid'] = silver_orders['delivery_time_mins'] < 0

print(f"Final silver_orders row count: {len(silver_orders):,}")
display(silver_orders.head(5))

Final silver_orders row count: 14,494


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,restaurant_name,cuisine,city,accepted,cancelled_ts,delivered_ts,picked_up,placed_ts,total_charged,total_refunded,net_revenue,delivery_time_mins,is_delivery_invalid
0,O000001,C01161,R052,2026-03-01 09:13:23+00:00,1.49,Old Burgers 52,Burgers,London,2026-03-01 09:15:28+00:00,NaT,2026-03-01 10:13:39+00:00,2026-03-01 09:37:00+00:00,2026-03-01 09:13:23+00:00,37.49,0.0,37.49,60.266667,False
1,O000002,C00723,R027,2026-03-01 09:15:33+00:00,2.49,Corner Chinese 27,Chinese,London,2026-03-01 09:21:19+00:00,NaT,2026-03-01 10:01:53+00:00,2026-03-01 09:39:13+00:00,2026-03-01 09:15:33+00:00,31.99,0.0,31.99,46.333333,False
2,O000003,C00781,R044,2026-03-01 09:23:42+00:00,0.00,Fresh Burgers 44,Burgers,Birmingham,2026-03-01 09:27:41+00:00,NaT,2026-03-01 10:02:45+00:00,2026-03-01 09:49:37+00:00,2026-03-01 09:23:42+00:00,12.50,0.0,12.50,39.050000,False
3,O000004,C01430,R049,2026-03-01 09:28:14+00:00,1.49,Happy Italian 49,Italian,Birmingham,2026-03-01 09:29:25+00:00,NaT,2026-03-01 10:10:41+00:00,2026-03-01 09:52:15+00:00,2026-03-01 09:28:14+00:00,46.99,0.0,46.99,42.450000,False
4,O000005,C00039,R009,2026-03-01 11:32:35+00:00,0.00,Green Italian 9,Italian,Birmingham,2026-03-01 11:36:36+00:00,2026-03-01 11:41:33+00:00,NaT,NaT,2026-03-01 11:32:35+00:00,6.00,-6.0,0.00,NaN,False


### **Step 3.5: Build Gold Layer (weekly_operations_report)**
We will group our successfully delivered orders by week (truncated to Monday), cuisine, and city. We calculate order count, net revenue, and average delivery times.

In [41]:
# Expected Result: A weekly operations dashboard with grouped rows (week starting Monday, cuisine, city) containing aggregated metrics.
# 1. Filter to delivered orders only (those with a delivered event)
gold_source = silver_orders[silver_orders['delivered_ts'].notnull()].copy()

# 2. Extract Week (starting Monday)
gold_source['week'] = gold_source['order_ts'].dt.to_period('W').dt.start_time

# 3. Handle delivery time exclusions: exclude negative or missing delivery durations
gold_source['valid_delivery_time'] = gold_source['delivery_time_mins'].where(~gold_source['is_delivery_invalid'])

# 4. Group and aggregate
gold_report = (
    gold_source.groupby(['week', 'cuisine', 'city'])
    .agg(
        total_orders=('order_id', 'count'),
        net_revenue=('net_revenue', 'sum'),
        avg_delivery_time_mins=('valid_delivery_time', 'mean')
    )
    .reset_index()
)

# Round average values for reporting elegance
gold_report['avg_delivery_time_mins'] = gold_report['avg_delivery_time_mins'].round(1)
gold_report['net_revenue'] = gold_report['net_revenue'].round(2)

print(f"Weekly Operations Dashboard (Gold layer) rows: {len(gold_report):,}")
display(gold_report.head(10))

Weekly Operations Dashboard (Gold layer) rows: 554


/tmp/ipykernel_2853/3226902681.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  gold_source['week'] = gold_source['order_ts'].dt.to_period('W').dt.start_time


,week,cuisine,city,total_orders,net_revenue,avg_delivery_time_mins
0,2026-02-23,Burgers,Birmingham,4,125.99,38.8
1,2026-02-23,Burgers,Bristol,3,66.77,40.2
2,2026-02-23,Burgers,London,11,396.42,39.7
3,2026-02-23,Chinese,Birmingham,6,117.46,34.0
4,2026-02-23,Chinese,London,7,284.93,35.1
5,2026-02-23,Chinese,Manchester,1,24.99,33.7
6,2026-02-23,Indian,Birmingham,1,43.99,42.6
7,2026-02-23,Indian,London,7,215.56,30.7
8,2026-02-23,Italian,Birmingham,7,138.94,36.2
9,2026-02-23,Italian,Leeds,1,49.49,32.0


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

### Part 1: The Order Volume Check

**What we are checking:** We want to make sure no delivered orders got lost when we grouped everything by week, city, and cuisine to make the final report.

**What to expect:** The total number of orders in our final dashboard should match the number of delivered orders we found in our clean master data (exactly **13,409** orders).

In [43]:
print("=== RECONCILIATION CHECKS ===\n")

# 1. Total Volume / Delivered Orders Check
gold_delivered_orders = gold_report['total_orders'].sum()
silver_delivered_count = silver_orders['delivered_ts'].notnull().sum()
orders_match = gold_delivered_orders == silver_delivered_count

print(f"1. Order Count Verification:")
print(f"  - Total orders in Gold report: {gold_delivered_orders:,}")
print(f"  - Total delivered orders in Silver: {silver_delivered_count:,}")
print(f"  - Match status: {'PASS' if orders_match else 'FAIL'}")
print()

=== RECONCILIATION CHECKS ===

1. Order Count Verification:
  - Total orders in Gold report: 13,409
  - Total delivered orders in Silver: 13,409
  - Match status: PASS



### Part 2: The Money Match Check

**What we are checking:** We want to make sure the cash adds up. We are checking if the total net revenue (charges minus refunds) in our final report matches the total money calculated from our cleaned transactions table.

**What to expect:** Both numbers should align down to the very last penny (exactly **£499,215.08**), with zero difference between them.

In [44]:
# 2. Financial / Revenue Reconciliation
gold_total_revenue = gold_report['net_revenue'].sum()
silver_delivered_revenue = silver_orders.loc[silver_orders['delivered_ts'].notnull(), 'net_revenue'].sum()
revenue_diff = abs(gold_total_revenue - silver_delivered_revenue)
revenue_match = revenue_diff < 1.0  # Allow minor rounding tolerance across 554 groups

print(f"2. Net Revenue Verification:")
print(f"  - Sum of net_revenue in Gold report: £{gold_total_revenue:,.2f}")
print(f"  - Sum of net_revenue for delivered in Silver: £{silver_delivered_revenue:,.2f}")
print(f"  - Absolute difference: £{revenue_diff:,.2f}")
print(f"  - Match status: {'PASS' if revenue_match else 'FAIL'}")
print()

2. Net Revenue Verification:
  - Sum of net_revenue in Gold report: £499,215.08
  - Sum of net_revenue for delivered in Silver: £499,215.08
  - Absolute difference: £0.00
  - Match status: PASS



### Part 3: The Delivery Time Speedometer Check

**What we are checking:** We want to see how much we can trust our average delivery time metrics. We count how many delivered orders have a valid, measurable delivery time duration (from placement to delivery) without any weird timestamp errors.

**What to expect:** Out of our 13,409 delivered orders, we expect exactly **16** orders to have broken timestamps (where delivery is recorded before placement). This should give us a clean, trustworthy coverage rate of **99.88%**.

In [45]:
# 3. Delivery Time Coverage and Sequence Check
delivered_orders_subset = silver_orders[silver_orders['delivered_ts'].notnull()]
total_delivered = len(delivered_orders_subset)
invalid_delivery_count = delivered_orders_subset['is_delivery_invalid'].sum()
valid_delivery_count = delivered_orders_subset['delivery_time_mins'].notnull().sum() - invalid_delivery_count

coverage_pct = (valid_delivery_count / total_delivered) * 100

print(f"3. Delivery Performance Coverage:")
print(f"  - Total delivered orders: {total_delivered:,}")
print(f"  - Out-of-sequence delivery anomalies (delivered < placed): {invalid_delivery_count}")
print(f"  - Delivered orders with valid/measurable durations: {valid_delivery_count:,}")
print(f"  - Metric trust coverage: {coverage_pct:.2f}%")
print()

3. Delivery Performance Coverage:
  - Total delivered orders: 13,409
  - Out-of-sequence delivery anomalies (delivered < placed): 16
  - Delivered orders with valid/measurable durations: 13,393
  - Metric trust coverage: 99.88%



### Part 4: Spot Checking a Relocating Restaurant (R007)

**What we are checking:** Since restaurant **R007** packed up and moved from Bristol to London in the middle of our timeline, we want to run a quick spot check to make sure our slowly-changing-dimension logic successfully captured both locations.

**What to expect:** We should see all 54 orders for R007 processed cleanly, with the city history showing both **Bristol** and **London** instead of just one.

In [46]:
# 4. Spot Check a Relocating Restaurant (R007)
r007_silver = silver_orders[silver_orders['restaurant_id'] == 'R007']
r007_gold_orders = gold_source[gold_source['restaurant_id'] == 'R007'].shape[0]

print(f"4. SCD Spot Check on Relocating Restaurant R007:")
print(f"  - Total orders processed for R007 in Silver: {len(r007_silver)}")
print(f"  - Total delivered (Gold source) orders for R007: {r007_gold_orders}")
print(f"  - Unique cities mapped for R007: {r007_silver['city'].unique()}")

4. SCD Spot Check on Relocating Restaurant R007:
  - Total orders processed for R007 in Silver: 54
  - Total delivered (Gold source) orders for R007: 50
  - Unique cities mapped for R007: ['Bristol' 'London']


Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

### How I worked with AI to build this:

* **What I prompted:** "Help me map orders to the correct city for restaurants that changed locations over time (SCD Type 2)."
* **What went wrong:** The AI initially tried to do a simple table join on `restaurant_id`, which caused a row fan-out and duplicated some of my orders.
* **How I fixed it:** I stepped in and corrected the code. I forced a merge, sorted the rows by their `effective_from` date, and used `.groupby().last()` to make sure each order was matched to exactly one correct historical location version.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

### What's still on the to-do list:

* **Checking the actual food items (The Order Lines):** We made sure the cash amounts match up, but we haven't actually checked if the individual items (burgers, ramen, salads) are correct inside each order or if there are any weird duplicate items in the lines.
* **The 16 out-of-sequence orders:** We found 16 orders where delivery is somehow recorded before placement. We excluded them to keep our speed metrics safe, but in a real-world scenario, we'd need to trace these IDs back to the database team to see if it's a timestamp glitch or a human typo.
* **Cancelled orders and refund details:** We completely skipped looking at cancelled orders and why they were cancelled. We'd love to build a 'refund rate' or 'cancel rate' metric next to see if certain cities or cuisines struggle more with order drops.
* **Customer address matches:** We haven't compared customer sign-up cities (`home_city`) to where they actually ordered from (`city`). Are customers ordering locally, or are they ordering food while travelling?

### If we had another 30 minutes, we would:
1. Write a quick automated alert system (like daily test checks) that flags future orders if they get processed out-of-order again.
2. Deep-dive into cancelled transactions to see if refunds are matching up perfectly for incomplete deliveries.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.